# Train the distillation teachers (GPU)

The ResNet-18 teachers for the KD runs. This notebook exists because the teacher is
**16.6x the student's cost per tile** — measured at 33.3 ms against 2.0 ms on four CPU
cores, so about **4.6 hours per seed on CPU** against minutes on a GPU. `CLAUDE.md`
specifies the teacher as Colab-only for exactly this reason.

**Before running:** Runtime → Change runtime type → **GPU** (a T4 is plenty).

What it does:

1. clones the repo and installs what is missing,
2. downloads DeepPCB and builds the tile dataset (~1 minute),
3. trains one teacher per seed on the frozen input mode (**C**, test + XOR),
4. checks each teacher **beats the student on validation** — if it does not, distilling
   from it is pointless and the KD runs should not be read,
5. zips the checkpoints for you to drop into `runs/teacher_s*/best.pt`.

Everything uses the repo's own `train.py`, so the teachers are trained by the same loop,
data, split and schedule as the students. That is deliberate: it makes "the teacher is
better" a statement about capacity and input resolution rather than about training setup.

## 1. Setup

In [ ]:
import os
import subprocess
import sys

import torch

BRANCH = "claude/lucid-cannon-eyj5yg"
REPO = "https://github.com/vsubacius/PCBerrorDetection.git"

if not os.path.isdir("PCBerrorDetection"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO], check=True)
os.chdir(os.path.abspath("PCBerrorDetection"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx"], check=False)

print("cwd  ", os.getcwd())
print("torch", torch.__version__)
print("GPU  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    print()
    print("!! No GPU detected. Runtime -> Change runtime type -> GPU, then rerun.")
    print("   On CPU each teacher takes about 4.6 hours instead of minutes.")

## 2. Dataset

Clones DeepPCB (~221 MB) and builds the bit-packed tile `.npz`. About a minute. The `.npz`
is only ~10 MB, so copy it to Drive if you want to skip this next time.

In [ ]:
import pathlib
import subprocess
import sys

NPZ = pathlib.Path("data/tiles/deeppcb_tiles_t160_s120.npz")
if not NPZ.exists():
    subprocess.run([sys.executable, "data/build_tiles.py", "--download",
                    "--root", "data/DeepPCB", "--out", "data/tiles"], check=True)
print("tile dataset:", NPZ, f"{NPZ.stat().st_size / 1e6:.0f} MB")

## 3. Train the teachers

Input mode **C** is frozen, from the step-8 input ablation: modes B and C tied within seed
noise, and C was chosen because Stage 1 already thresholds the eroded XOR, so both stages
see the same difference signal.

`batch_images=8` is 8 boards x 25 tiles = 200 tiles per step. Raising it does not change
the result, only the speed.

In [ ]:
import sys
import time

sys.path.insert(0, "training")
from train import TrainConfig, train

SEEDS = (0, 1, 2)
REFERENCE_MODE = "C"

results = {}
for seed in SEEDS:
    config = TrainConfig(
        run_id=f"teacher_s{seed}",
        model="teacher",
        representation=REFERENCE_MODE,
        loss_kind="plain",
        seed=seed,
        epochs=30,
        batch_images=8,
        num_workers=2,
        device="cuda" if torch.cuda.is_available() else "cpu",
        out_dir="runs",
    )
    started = time.time()
    results[config.run_id] = train(config)
    print(f"--- {config.run_id} done in {(time.time() - started) / 60:.1f} min ---")
    print()

## 4. Does the teacher actually beat the student?

**This is the check that matters.** 11.2M parameters trained on 20,000 tiles drawn from 800
images — themselves only 7 physical boards — will overfit hard. A teacher that does not beat
the student on validation has nothing to teach it, and any KD result read off such a teacher
would be meaningless.

The student baselines below are the measured `input_C` runs (mode C, plain BCE, three
seeds): macro AP **0.8050 ± 0.0333**.

In [ ]:
import json
import pathlib
import statistics

# Measured input_C runs, from runs/summary_input_ablation.json.
STUDENT_AP = {0: 0.8423241201137754, 1: 0.7942767751604629, 2: 0.7784492343173023}
STUDENT_SPREAD = 0.0333

rows = []
for seed in SEEDS:
    path = pathlib.Path(f"runs/teacher_s{seed}/metrics.json")
    if not path.exists():
        print(f"teacher_s{seed}: no metrics.json, skipped")
        continue
    rows.append((seed, json.loads(path.read_text())["best_val_macro_ap"],
                 STUDENT_AP[seed]))

print(f"{'seed':>5}{'teacher mAP':>14}{'student mAP':>14}{'margin':>10}")
print("-" * 43)
for seed, teacher_ap, student_ap in rows:
    print(f"{seed:>5}{teacher_ap:>14.4f}{student_ap:>14.4f}{teacher_ap - student_ap:>+10.4f}")

if rows:
    t_mean = statistics.mean(r[1] for r in rows)
    s_mean = statistics.mean(r[2] for r in rows)
    margin = t_mean - s_mean
    print()
    print(f"mean teacher {t_mean:.4f}   mean student {s_mean:.4f}   margin {margin:+.4f}")
    print()
    if margin <= 0:
        print("!! The teacher does NOT beat the student.")
        print("   Do not run the KD comparisons on these checkpoints: there is nothing")
        print("   to distil. Report it -- that is a real result about capacity on 800")
        print("   images, not a bug. Options: stronger augmentation, dropout, fewer")
        print("   epochs, or a smaller teacher.")
    elif margin < STUDENT_SPREAD:
        print(f"!  The margin is inside the student's own seed spread (+/-{STUDENT_SPREAD}),")
        print("   so the teacher is not clearly better. KD has little headroom here, and")
        print("   the KD result should say so.")
    else:
        print("OK: the teacher is meaningfully better, so the KD runs have headroom.")

## 5. Download the checkpoints

Each teacher is ~45 MB, which is why they are **not** committed to the repo. Unzip into the
repo root so the files land at `runs/teacher_s*/best.pt`, then run:

```
python training/run_matrix.py --set kd
```

`run_matrix` refuses to start a KD run whose teacher checkpoint is missing, so it will tell
you if a file did not land where it should.

In [ ]:
import glob
import os
import zipfile

paths = sorted(glob.glob("runs/teacher_s*/best.pt"))
paths += sorted(glob.glob("runs/teacher_s*/metrics.json"))
paths += sorted(glob.glob("runs/teacher_s*/config.json"))

if not paths:
    raise SystemExit("nothing to package -- did section 3 run?")

print("packaging:")
for path in paths:
    print(f"  {path}  {os.path.getsize(path) / 1e6:.1f} MB")

archive = "teachers.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in paths:
        bundle.write(path, path)
print()
print(f"{archive}  {os.path.getsize(archive) / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(archive)
except Exception as exc:
    print(f"download it from the file browser instead ({exc})")